In [2]:
#coding:gbk
r"""
基于 AkShare 的盘后 ETF 资金护盘状态采集器 (本地直连版)

去除了原有的 HTTP API 依赖，直接通过 akshare 开源接口获取真实市场数据。
默认监测沪深 300 (510300) 和上证 50 (510050) 的极端放量护盘动作。

环境变量配置:
    NATIONAL_TEAM_ETFS
        需要监测的 ETF 列表，逗号分隔。默认: "510300,510050"
    QMT_REGIME_DIR
        状态文件输出目录。默认: C:\qmt_regime
    REGIME_RUN_TIME
        守护进程运行时间。默认: 15:35
"""

import argparse
import csv
import datetime
import json
import math
import os
import tempfile
import time
import sys
import akshare as ak
import pandas as pd

OUTPUT_FIELDS = [
    "trade_date",
    "regime",
    "direction",
    "total_net_flow_cny",
    "total_aum_cny",
    "flow_ratio_bps",
    "flow_zscore",
    "strength_score",
    "etf_count",
    "generated_at",
]

def fetch_etf_data_via_akshare(trade_date_iso, allowlist):
    """
    使用 AkShare 获取 ETF 数据，并构建护盘因子的特征输入。
    """
    date_str = trade_date_iso.replace("-", "")
    normalized_records = []
    
    print(f"正在通过 AkShare 拉取 {trade_date_iso} 的行情数据...")
    
    try:
        spot_df = ak.fund_etf_spot_em()
    except Exception as e:
        raise RuntimeError(f"获取 ETF 实时概况失败: {e}")

    for symbol in allowlist:
        pure_code = symbol.split(".")[0] if "." in symbol else symbol
        
        try:
            hist_df = ak.fund_etf_hist_em(
                symbol=pure_code, 
                period="daily", 
                start_date=date_str, 
                end_date=date_str
            )
            
            if hist_df.empty:
                print(f"警告: {symbol} 在 {trade_date_iso} 无交易数据。")
                continue
                
            amount_cny = float(hist_df["成交额"].iloc[0])
            pct_change = float(hist_df["涨跌幅"].iloc[0])
            
            aum_cny = 0.0
            match_spot = spot_df[spot_df["代码"] == pure_code]
            if not match_spot.empty:
                aum_cny = float(match_spot["总市值"].iloc[0])
                
            proxy_inflow_cny = amount_cny
            
            normalized_records.append({
                "symbol": symbol,
                "net_inflow_cny": proxy_inflow_cny,
                "aum_cny": aum_cny
            })
            print(f"  > 成功拉取 {symbol} | 成交额: {proxy_inflow_cny/1e8:.2f}亿 | 涨跌幅: {pct_change}%")
            
        except Exception as e:
            print(f"拉取 {symbol} 数据时出错: {e}")
            continue

    if not normalized_records:
        raise ValueError("未能通过 AkShare 获取到任何有效的 ETF 数据。")

    return trade_date_iso, normalized_records, None

def load_history(path):
    if not os.path.exists(path):
        return []
    rows = []
    with open(path, "r") as handle:
        for row in csv.DictReader(handle):
            try:
                row["total_net_flow_cny"] = float(row["total_net_flow_cny"])
                rows.append(row)
            except Exception:
                continue
    rows.sort(key=lambda item: item["trade_date"])
    return rows

def flow_zscore(history, current_flow, lookback=60, min_history=20):
    values = [float(row["total_net_flow_cny"]) for row in history[-lookback:]]
    if len(values) < min_history:
        return 0.0, False
    mean = sum(values) / float(len(values))
    variance = sum((value - mean) ** 2 for value in values) / float(len(values))
    std = math.sqrt(variance)
    if std <= 0:
        return 0.0, True
    return (current_flow - mean) / std, True

def classify_regime(records, history, provider_strong, abs_flow_threshold, ratio_bps_threshold, zscore_threshold):
    total_flow = sum(item["net_inflow_cny"] for item in records)
    total_aum = sum(item["aum_cny"] for item in records)
    flow_ratio_bps = (total_flow / total_aum * 10000.0 if total_aum > 0 else 0.0)
    zscore, statistically_ready = flow_zscore(history, total_flow)

    magnitude_score = abs(total_flow) / max(abs_flow_threshold, 1.0)
    ratio_score = abs(flow_ratio_bps) / max(ratio_bps_threshold, 1.0)
    zscore_score = abs(zscore) / max(zscore_threshold, 0.01)
    strength_score = max(magnitude_score, ratio_score, zscore_score)

    magnitude_gate = abs(total_flow) >= abs_flow_threshold
    ratio_gate = total_aum > 0 and abs(flow_ratio_bps) >= ratio_bps_threshold
    statistical_gate = statistically_ready and abs(zscore) >= zscore_threshold

    if isinstance(provider_strong, bool):
        strong = provider_strong
    else:
        strong = ratio_gate or (magnitude_gate and (statistical_gate or not statistically_ready))

    direction = "INFLOW" if total_flow > 0 else "FLAT"

    return {
        "regime": "INTERVENTION" if strong else "NORMAL",
        "direction": direction,
        "total_net_flow_cny": total_flow,
        "total_aum_cny": total_aum,
        "flow_ratio_bps": flow_ratio_bps,
        "flow_zscore": zscore,
        "strength_score": strength_score,
        "etf_count": len(records),
    }

def atomic_write_json(path, data):
    directory = os.path.dirname(path)
    if not os.path.isdir(directory):
        os.makedirs(directory)
    fd, temp_path = tempfile.mkstemp(prefix="regime_", suffix=".json", dir=directory)
    try:
        with os.fdopen(fd, "w") as handle:
            json.dump(data, handle, indent=2, sort_keys=True)
        os.replace(temp_path, path)
    except Exception:
        try:
            os.unlink(temp_path)
        except Exception:
            pass
        raise

def atomic_write_history(path, rows):
    directory = os.path.dirname(path)
    if not os.path.isdir(directory):
        os.makedirs(directory)
    fd, temp_path = tempfile.mkstemp(prefix="regime_", suffix=".csv", dir=directory)
    try:
        with os.fdopen(fd, "w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=OUTPUT_FIELDS)
            writer.writeheader()
            for row in rows:
                writer.writerow({field: row.get(field, "") for field in OUTPUT_FIELDS})
        os.replace(temp_path, path)
    except Exception:
        try:
            os.unlink(temp_path)
        except Exception:
            pass
        raise

def run_once(trade_date=None):
    trade_date = trade_date or datetime.date.today().isoformat()
    # 修复：加上 r 表示原生字符串，防止 \q 报错
    output_dir = os.environ.get("QMT_REGIME_DIR", r"C:\qmt_regime")
    
    default_etfs = "510300.SH,510050.SH"
    allowlist = set(
        symbol.strip().upper()
        for symbol in os.environ.get("NATIONAL_TEAM_ETFS", default_etfs).split(",")
        if symbol.strip()
    )

    abs_flow_threshold = float(os.environ.get("REGIME_ABS_FLOW_THRESHOLD_CNY", "5000000000"))
    ratio_bps_threshold = float(os.environ.get("REGIME_FLOW_RATIO_BPS_THRESHOLD", "20"))
    zscore_threshold = float(os.environ.get("REGIME_ZSCORE_THRESHOLD", "2.0"))

    json_path = os.path.join(output_dir, "national_team_regime.json")
    history_path = os.path.join(output_dir, "national_team_regime_history.csv")
    
    history = [row for row in load_history(history_path) if row["trade_date"] != trade_date]

    actual_date, records, provider_strong = fetch_etf_data_via_akshare(trade_date, allowlist)
    
    history = [row for row in history if row["trade_date"] != actual_date]
    metrics = classify_regime(records, history, provider_strong, abs_flow_threshold, ratio_bps_threshold, zscore_threshold)

    output = {
        "trade_date": actual_date,
        "generated_at": datetime.datetime.now().isoformat(),
        "regime": metrics["regime"],
        "direction": metrics["direction"],
        "total_net_flow_cny": metrics["total_net_flow_cny"],
        "total_aum_cny": metrics["total_aum_cny"],
        "flow_ratio_bps": metrics["flow_ratio_bps"],
        "flow_zscore": metrics["flow_zscore"],
        "strength_score": metrics["strength_score"],
        "etf_count": metrics["etf_count"],
        "symbols": [item["symbol"] for item in records],
    }

    history.append(output.copy())
    history.sort(key=lambda item: item["trade_date"])
    atomic_write_history(history_path, history)
    atomic_write_json(json_path, output)
    
    print("\n--- 判定结果 ---")
    print(json.dumps(output, sort_keys=True, ensure_ascii=False, indent=2))
    return output

def run_daemon():
    run_time = os.environ.get("REGIME_RUN_TIME", "15:35")
    last_run_date = None
    print(f"AkShare Collector 守护进程已启动; 设定的执行时间为每天 {run_time}")
    while True:
        now = datetime.datetime.now()
        today = now.date().isoformat()
        if (now.weekday() < 5 and now.strftime("%H:%M") >= run_time and last_run_date != today):
            try:
                run_once(today)
                last_run_date = today
            except Exception as exc:
                print(f"Collector error: {exc}")
        time.sleep(30)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--date", default=None, help="YYYY-MM-DD")
    parser.add_argument("--daemon", action="store_true", help="开启守护进程模式，每日盘后自动运行")
    
    # 修复：检测是否在 Jupyter 中运行，如果是，则清空隐藏参数
    if 'ipykernel' in sys.modules:
        args = parser.parse_args(args=[])
    else:
        args = parser.parse_args()
        
    if args.daemon:
        run_daemon()
    else:
        test_date = "2026-08-04" 
        print(f">>> 正在使用测试日期: {test_date} 运行...")
        run_once(test_date)

if __name__ == "__main__":
    main()

>>> 正在使用测试日期: 2026-08-04 运行...
正在通过 AkShare 拉取 2026-08-04 的行情数据...


  0%|          | 0/15 [00:00<?, ?it/s]

拉取 510300.SH 数据时出错: HTTPSConnectionPool(host='push2his.eastmoney.com', port=443): Max retries exceeded with url: /api/qt/stock/kline/get?fields1=f1%2Cf2%2Cf3%2Cf4%2Cf5%2Cf6&fields2=f51%2Cf52%2Cf53%2Cf54%2Cf55%2Cf56%2Cf57%2Cf58%2Cf59%2Cf60%2Cf61%2Cf116&ut=7eea3edcaed734bea9cbfc24409ed989&klt=101&fqt=0&beg=20260804&end=20260804&secid=1.510300 (Caused by ProxyError('Unable to connect to proxy', RemoteDisconnected('Remote end closed connection without response')))
拉取 510050.SH 数据时出错: HTTPSConnectionPool(host='push2his.eastmoney.com', port=443): Max retries exceeded with url: /api/qt/stock/kline/get?fields1=f1%2Cf2%2Cf3%2Cf4%2Cf5%2Cf6&fields2=f51%2Cf52%2Cf53%2Cf54%2Cf55%2Cf56%2Cf57%2Cf58%2Cf59%2Cf60%2Cf61%2Cf116&ut=7eea3edcaed734bea9cbfc24409ed989&klt=101&fqt=0&beg=20260804&end=20260804&secid=1.510050 (Caused by ProxyError('Unable to connect to proxy', RemoteDisconnected('Remote end closed connection without response')))


ValueError: 未能通过 AkShare 获取到任何有效的 ETF 数据。

In [1]:
import time
import pandas as pd
from xtquant import xtdata
from xtquant.xttrader import XtQuantTrader, XtQuantTraderCallback
from xtquant import xtconstant
from xtquant.xttype import StockAccount
import datetime

# ==========================================
# [配置区] 请务必修改为您本地的真实参数
# ==========================================
MINI_QMT_PATH = r'C:\Users\baby liu\Desktop\中信建投\中信建投证券QMT仿真-交易端\userdata_mini' 
ACCOUNT_ID = '34165036'                      
ACTION_POINT = 12.0                                 # 触发套利的利润阈值 
FEE_ESTIMATE = 6.0                                  # 预估的4条腿总交易成本/滑点折损 (元)
POLL_INTERVAL = 0.5                                 # 盘口轮询间隔 (秒)

# 定义要监控的盒式套利合约组 (示例为上证50ETF期权)
# K1: 低行权价, K2: 高行权价
BOX_CONFIG = {
    'K1': 1600,
    'K2': 1700,
    'call_k1': '10011031.SH',  # K1 认购
    'put_k1':  '10011040.SH',  # K1 认沽
    'call_k2': '10011028.SH',  # K2 认购
    'put_k2':  '10011034.SH',  # K2 认沽
    'multiplier': 10000        # 合约乘数
}

# 需要订阅的合约列表
CONTRACT_LIST = [
    BOX_CONFIG['call_k1'], BOX_CONFIG['put_k1'], 
    BOX_CONFIG['call_k2'], BOX_CONFIG['put_k2']
]

# ==========================================
# 1. 交易回调类 (接收成交回报)
# ==========================================
class MyXtQuantCallback(XtQuantTraderCallback):
    def on_disconnected(self):
        print("[-] MiniQMT 交易终端连接断开")

    def on_stock_order(self, order):
        print(f"[委托回报] 合约: {order.stock_code}, 状态: {order.order_status}, 报单价: {order.price}")

    def on_stock_trade(self, trade):
        print(f"[成交回报] 合约: {trade.stock_code}, 成交价: {trade.traded_price}, 数量: {trade.traded_volume}")

    def on_stock_error(self, error):
        print(f"[交易错误] {error.error_msg}")

# ==========================================
# 2. 策略核心逻辑 (基于实时盘口)
# ==========================================
def check_box_arbitrage():
    """
    拉取五档行情并计算盒式套利空间
    返回: orders (订单列表), profit (净利润)
    """
    # 拉取最新快照
    tick_dict = xtdata.get_full_tick(CONTRACT_LIST)
    
    # 确保4个合约都有数据返回
    if len(tick_dict) < 4:
        return [], 0
        
    try:
        # 获取盘口买一/卖一价 (对价交易，保证立刻成交)
        # 盘口买价(Bid)是用来卖出的，盘口卖价(Ask)是用来买入的
        ask1_c1 = tick_dict[BOX_CONFIG['call_k1']]['askPrice'][0]  # 买入 Call K1 付出的价格
        bid1_p1 = tick_dict[BOX_CONFIG['put_k1']]['bidPrice'][0]   # 卖出 Put K1 收到的价格
        bid1_c2 = tick_dict[BOX_CONFIG['call_k2']]['bidPrice'][0]  # 卖出 Call K2 收到的价格
        ask1_p2 = tick_dict[BOX_CONFIG['put_k2']]['askPrice'][0]   # 买入 Put K2 付出的价格
        
        # 排除盘口为空 (价格为0) 的情况，防止跌停/涨停导致计算异常
        if ask1_c1 == 0 or bid1_p1 == 0 or bid1_c2 == 0 or ask1_p2 == 0:
            return [], 0

        # 计算每份合约的建仓总成本
        cost_per_unit = ask1_c1 - bid1_p1 - bid1_c2 + ask1_p2
        
        # 盒式套利到期理论收益
        payoff_per_unit = BOX_CONFIG['K2'] - BOX_CONFIG['K1']
        
        # 计算单张合约的净利润 (乘以乘数，扣除摩擦成本)
        profit = (payoff_per_unit - cost_per_unit) * BOX_CONFIG['multiplier'] - FEE_ESTIMATE
        
        if profit >= ACTION_POINT:
            # 构造订单队列
            orders = [
                # 动作 1: 买入开仓 Call K1
                {'code': BOX_CONFIG['call_k1'], 'type': xtconstant.OPT_BUY_OPEN,  'price': ask1_c1},
                # 动作 2: 卖出开仓 Put K1
                {'code': BOX_CONFIG['put_k1'],  'type': xtconstant.OPT_SELL_OPEN, 'price': bid1_p1},
                # 动作 3: 卖出开仓 Call K2
                {'code': BOX_CONFIG['call_k2'], 'type': xtconstant.OPT_SELL_OPEN, 'price': bid1_c2},
                # 动作 4: 买入开仓 Put K2
                {'code': BOX_CONFIG['put_k2'],  'type': xtconstant.OPT_BUY_OPEN,  'price': ask1_p2}
            ]
            return orders, profit
            
    except Exception as e:
        print(f"数据解析异常: {e}")
        
    return [], 0

# ==========================================
# 3. 订单执行引擎
# ==========================================
def execute_box_orders(orders, trader, account):
    """
    并发向 QMT 发送 4 条腿的订单
    """
    for order in orders:
        seq = trader.order_stock_async(
            account=account,
            stock_code=order['code'],
            order_type=order['type'],
            order_volume=1,  # 默认每次下单 1 张
            price_type=xtconstant.FIX_PRICE,
            price=order['price'],
            strategy_name="Box_Arbitrage_Live",
            order_remark="盒式套利发单"
        )
        print(f"[+] 发送指令 | 流水号: {seq} | 合约: {order['code']} | 价格: {order['price']}")

# ==========================================
# 主程序入口
# ==========================================
if __name__ == '__main__':
    print("="*40)
    print("启动 MiniQMT 期权盒式套利系统...")
    print("="*40)
    
    # 1. 初始化交易终端
    session_id = int(time.time())
    xt_trader = XtQuantTrader(MINI_QMT_PATH, session_id)
    
    # 2. 注册回调并启动后台线程
    xt_trader.register_callback(MyXtQuantCallback())
    xt_trader.start()
    
    # 3. 建立连接
    print(f"[*] 正在连接 QMT 终端, 路径: {MINI_QMT_PATH}")
    connect_result = xt_trader.connect()
    
    if connect_result == 0:
        print("[+] 连接成功！")
    else:
        print(f"[-] 连接失败，错误码: {connect_result}")
        print("请检查：1. QMT是否已启动并登录; 2. 是否开启极简模式; 3. 路径和账号是否正确。")
        import sys; sys.exit()
        
    # 4. 订阅账号
    account = StockAccount(ACCOUNT_ID, 'STOCK')
    xt_trader.subscribe(account)
    
    # 5. 订阅行情数据 (持续接收 Tick)
    print(f"[*] 正在订阅行情: {CONTRACT_LIST}")
    for code in CONTRACT_LIST:
        xtdata.subscribe_quote(code, period='tick', count=-1)
        
    # 给 QMT 几秒钟时间缓冲行情初始数据
    time.sleep(3) 

    # 6. 进入实盘高频轮询主循环
    print("[*] 进入监控模式，等待套利空间出现...")
    try:
        while True:
            # 检查信号
            orders, profit = check_box_arbitrage()
            
            if orders:
                print("\n" + "="*40)
                print(f"!!! 发现套利机会 !!!")
                print(f"当前无风险利润: {profit:.2f} 元 (阈值: {ACTION_POINT})")
                print("="*40)
                
                # 执行下单
                execute_box_orders(orders, xt_trader, account)
                
                # 下单后休眠一段时间，防止因未成交导致的重复疯狂发单
                # 实盘中这里通常需要加入订单状态检查逻辑 (未成交则撤单)
                print("[*] 发单完毕，休眠 10 秒防重发...")
                time.sleep(10) 
                
            # 控制轮询频率，降低 CPU 占用
            time.sleep(POLL_INTERVAL)
            
    except KeyboardInterrupt:
        print("\n[*] 接收到停止指令，正在退出系统...")
        xt_trader.stop()

ModuleNotFoundError: No module named 'xtquant'